In [ ]:
#| default_exp docsprocs

# docsprocs

> A docs-build notebook processor that color-codes cells by boopiter type, so the rendered site echoes the app's colored left bars.

Registered via `doc_procs` in `pyproject.toml [tool.nbdev]`, `color_cells` runs over every notebook during the docs build (before Quarto renders). It wraps each markdown/raw cell's source in a Quarto fenced div classed by its boopiter cell type (`.boop-note` green, `.boop-prompt` red, `.boop-raw` orange); `styles.css` turns those into the colored left bar. Code cells already carry `.cell-code`, so they're colored in CSS alone. The page's H1 title cell is left untouched so Quarto's title handling isn't disturbed.

In [ ]:
#| export
import re

# a Prompt+Reply markdown cell (solveit encoding) carries this separator -- see serialize.py
_SEP_RE = re.compile(r'##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_[0-9a-f]+ -->')

def color_cells(cell):
    "nbdev docs processor: wrap a markdown/raw cell's source in a Quarto fenced div classed by its boopiter cell type, so styles.css can draw boopiter's colored left bar (green note / red prompt / orange raw). Code cells carry `.cell-code` and are colored via CSS; the page-title (H1) cell is left alone."
    t = cell.get('cell_type'); src = cell.get('source') or ''
    if not src.strip(): return
    first = src.lstrip().splitlines()[0]
    if t == 'raw': cls = 'boop-raw'
    elif t == 'markdown':
        if first.startswith('# '): return          # leave the page-title (H1) cell alone
        cls = 'boop-prompt' if _SEP_RE.search(src) else 'boop-note'
    else: return                                    # code cells are handled in CSS
    cell['source'] = f'::: {{.boopcell .{cls}}}\n{src}\n:::'


In [ ]:
#| export
import hashlib, json, os, shutil, socket, subprocess, tempfile, textwrap
from datetime import date
from pathlib import Path

SHARE_REPO    = 'boops'                            # GitHub repo whose gh-pages branch serves the shared notebooks
# Local Quarto project holding every share; only its _site is ever published. Deliberately NOT under
# a dot-directory like ~/.boopiter: Quarto skips dot-prefixed paths when discovering a project's input
# files, so a project rooted inside one finds zero documents and renders nothing (silently -- publish
# still succeeds, having pushed an empty site). Sits with the user's other repos when that layout
# exists, since it is a real git repo, and is shared by every boopiter instance regardless of the
# directory the server was started in.
SHARE_PROJECT = (Path.home()/'github'/SHARE_REPO) if (Path.home()/'github').is_dir() else (Path.home()/SHARE_REPO)
# Assets the docs build keeps beside its generated _quarto.yml. Copied as-is rather than regenerated:
# theme-toggle.html + styles.css are what give a page boopiter's sun/moon toggle and coloured cell
# bars, and they only stay in step with the docs site by being the same files.
_SHARE_ASSETS = ('styles.css', 'booptheme.scss', 'theme-toggle.html')
# Card metadata for one published page, left beside it on gh-pages. That branch is the only thing two
# machines have in common -- notebook sources never leave the machine that made them -- so this is how a
# machine puts a page it cannot render onto the index. See _remote_shares and _index_contents.
_SIDECAR = 'share.json'
_INDEX_QMD = """---
title: "boops"
subtitle: "Notebooks shared from [boopiter]({docs})"
listing:
  contents:{contents}
  type: grid
  grid-columns: 3
  sort: "date desc"
  # 'image' has to be listed explicitly: fields is a whitelist, and leaving it out suppresses the
  # thumbnail even though Quarto has already found one -- it derives a preview from the first image in
  # each document (the same discovery behind og:image), so nothing needs linking by hand.
  fields: [image, title, description, date]
  date-format: "YYYY-MM-DD"
---
"""

def _repo_root() -> Path:
    "The boopiter checkout this module was imported from -- where _proc/ and images/ live."
    return Path(__file__).parent.parent if '__file__' in globals() else Path.cwd()

def share_slug(path) -> str:
    "Six hex chars identifying a notebook by machine and location: sha256 of hostname + its canonical path. Deterministic, so re-sharing the same file overwrites the same page and a link you've already sent stays current -- that's the whole point of not using a random suffix. Keyed on more than the basename because 'example.ipynb' in two checkouts, or on two machines, are different documents that would otherwise fight over one URL. realpath (not abspath) so two symlinked routes to one file still collapse to a single page, and a NUL separator so host 'a' + path 'b/c' can't collide with host 'a/b' + path 'c'."
    key = f"{socket.gethostname()}\0{os.path.realpath(path)}"
    return hashlib.sha256(key.encode()).hexdigest()[:6]

def share_name(path) -> str:
    "The published page's directory name for a notebook: '<basename>_<slug>' (see share_slug)."
    return f'{Path(path).stem}_{share_slug(path)}'

def _gh_owner() -> str:
    "The GitHub login that owns the share repo, from the authenticated gh CLI."
    r = subprocess.run(['gh', 'api', 'user', '--jq', '.login'], capture_output=True, text=True, timeout=30, check=True)
    return r.stdout.strip()

def _write_project_config(proc:Path, out:Path) -> None:
    "Write the share site's _quarto.yml from the docs build's generated one, adjusting only what genuinely differs. Deliberately NOT hand-written: theme, highlight styles, css and include-after-body all have to match the docs site, and the only way they stay matched is by being the same file. What changes: nbdev's pre/post-render steps (they build apilist/llms.txt for the full site), the sidebar and search (they index pages that aren't here), and sidebar.yml (generated per-site, absent from _proc). `drafts: unlinked` is what makes an unlisted share work -- Quarto still renders a draft page, it just keeps it out of listings and the sitemap, so the URL works but nothing points at it."
    import yaml
    cfg = yaml.safe_load((proc/'_quarto.yml').read_text())
    proj = cfg.setdefault('project', {})
    for k in ('pre-render', 'post-render', 'resources'): proj.pop(k, None)
    proj['type'], proj['output-dir'] = 'website', '_site'
    # No metadata-files at all: nbdev.yml is the only one present, and merging it would drag in the
    # docs project's own settings -- notably output-dir: _docs, which silently overrides the output-dir
    # set just above, so the render lands somewhere publish isn't looking. site-url is read from it
    # directly instead (see _ensure_share_project).
    cfg.pop('metadata-files', None)
    site = cfg.setdefault('website', {})
    site['sidebar'] = False
    site['drafts'] = 'unlinked'          # unlisted shares: rendered and reachable, absent from the listing
    site['title'] = 'boops'
    site.pop('site-url', None)           # inherited from nbdev.yml and would point at the docs site
    nav = site.setdefault('navbar', {})
    nav['search'] = False                # navbar stays: it carries Quarto's colour-scheme toggle
    # No logo-href/title-href: left unset, Quarto points the 'boops' brand at the site root, which is
    # the boops index -- where you want to land from a shared page. (The docs site is linked from the
    # index's own subtitle instead.)
    logo = _repo_root()/'images'/'logo.png'
    if logo.exists():
        shutil.copy(logo, out/'logo.png')
        site['favicon'] = 'logo.png'
    (out/'_quarto.yml').write_text(yaml.safe_dump(cfg, sort_keys=False))

_CRED_HELPER = 'credential.https://github.com.helper'   # lets gh hand git its token; see _ensure_git_repo

def _ensure_git_repo() -> None:
    "Give SHARE_PROJECT what `quarto publish gh-pages` needs of a local repo: the repo itself, the share remote, a commit identity, and a credential to push with. Every step is keyed on what is actually missing and is safe to repeat, so a share that dies halfway through setup is repaired by the next one instead of poisoning it. What this replaces ran the whole bootstrap only when the project directory didn't exist -- a condition `mkdir` invalidates two lines later, so anything failing after that point (gh not logged in yet, most likely) left a directory with no repo inside it, and every later share then saw the directory, skipped setup, and died deep inside quarto with 'not a git repository', which points at nothing. Identity and credential go in the repo's own config, and only when git can't already answer for itself, so a configured ~/.gitconfig still wins: quarto shells out to git on its own, so there is no handing it `-c` overrides, and a freshly installed machine has neither an identity (the commit fails outright) nor any way to authenticate the https push."
    owner = _gh_owner()
    g = ['git', '-C', str(SHARE_PROJECT)]
    if subprocess.run(['gh', 'repo', 'view', f'{owner}/{SHARE_REPO}'], capture_output=True).returncode:
        subprocess.run(['gh', 'repo', 'create', f'{owner}/{SHARE_REPO}', '--public',
                        '-d', 'Notebooks shared from boopiter'], capture_output=True, timeout=60, check=True)
    if not (SHARE_PROJECT/'.git').is_dir():
        subprocess.run(g + ['init', '-q', '-b', 'main'], check=True, timeout=60)
    # add first, then set-url: `add` fails once origin exists, which is every run after the first, while
    # a remote left pointing somewhere stale would quietly publish the site to the wrong repo.
    url = f'https://github.com/{owner}/{SHARE_REPO}.git'
    subprocess.run(g + ['remote', 'add', 'origin', url], capture_output=True, timeout=60)
    subprocess.run(g + ['remote', 'set-url', 'origin', url], capture_output=True, timeout=60)
    # `git var` resolves an identity from config or environment and exits non-zero when there is none to
    # find -- a better test than reading user.email, since an identity can come from either.
    if subprocess.run(g + ['var', 'GIT_COMMITTER_IDENT'], capture_output=True).returncode:
        subprocess.run(g + ['config', 'user.name', owner], check=True, timeout=60)
        subprocess.run(g + ['config', 'user.email', f'{owner}@users.noreply.github.com'], check=True, timeout=60)
    # The remote is https, so the push needs a github.com credential. gh already holds one and can serve
    # it as a helper, which makes an authenticated gh the only requirement -- no `gh auth setup-git` step
    # for the user to have missed, and nothing written outside this repo.
    if not subprocess.run(g + ['config', '--get', _CRED_HELPER], capture_output=True).stdout.strip():
        subprocess.run(g + ['config', _CRED_HELPER, '!gh auth git-credential'], check=True, timeout=60)
    _ensure_gh_pages_branch(owner)

def _ensure_share_project() -> Path:
    "The local Quarto project every share is rendered inside, created on first use. One project rather than a throwaway per share is the whole point: Quarto then emits a single site_libs (Bootstrap, theme CSS, its JS) and one favicon for the site, instead of a ~2MB copy per notebook. It's a git repo with the share repo as its remote, because that's what `quarto publish gh-pages` pushes through -- but only the rendered _site is ever published, so notebook sources stay on this machine."
    proc = _repo_root()/'_proc'
    if not (proc/'_quarto.yml').exists():
        raise RuntimeError(f'{proc}/_quarto.yml not found -- run nbdev_docs once so the docs config exists')
    SHARE_PROJECT.mkdir(parents=True, exist_ok=True)
    # Assets come from nbs/, the source of truth -- _proc holds copies that only refresh when the docs
    # are rebuilt, so taking them from there meant a theme edit didn't reach a share until then.
    nbs = _repo_root()/'nbs'
    for a in _SHARE_ASSETS:
        src = nbs/a if (nbs/a).exists() else proc/a
        if src.exists(): shutil.copy(src, SHARE_PROJECT/a)
    _write_project_config(proc, SHARE_PROJECT)
    _write_index(SHARE_PROJECT, {})   # rewritten with the gh-pages union before each render; see share_notebook
    # Nothing here is ever committed -- the branch we publish to is built by _publish_additive, and the
    # notebook sources sitting in this project are exactly what must not be pushed anywhere. The list
    # covers Quarto's byproducts too (keep-md output, the listing cache) so a stray `git add -A` in here
    # can't sweep a render into a commit.
    (SHARE_PROJECT/'.gitignore').write_text('_site/\n_docs/\n_freeze/\n.quarto/\nsite_libs/\nindex.html.md\n'
                                            'index-listing.json\n*/index.html.md\n')
    _ensure_git_repo()
    return SHARE_PROJECT

def _ensure_gh_pages_branch(owner:str) -> None:
    "Create an empty gh-pages branch on the remote if it has none, and point Pages at it.  refuses to bootstrap the branch itself under --no-prompt (it only offers to when run interactively), so we make it here with plumbing -- commit-tree over an empty tree -- which needs no checkout and so can't disturb the project's own working tree."
    g = ['git', '-C', str(SHARE_PROJECT)]
    if subprocess.run(g + ['ls-remote', '--exit-code', '--heads', 'origin', 'gh-pages'], capture_output=True).returncode:
        tree = subprocess.run(g + ['mktree'], input=b'', capture_output=True, check=True).stdout.strip().decode()
        commit = subprocess.run(g + ['-c', 'user.email=boopiter@localhost', '-c', 'user.name=boopiter',
                                     'commit-tree', tree, '-m', 'Initialize gh-pages'],
                                capture_output=True, check=True).stdout.strip().decode()
        subprocess.run(g + ['push', 'origin', f'{commit}:refs/heads/gh-pages'], capture_output=True, check=True, timeout=120)
    subprocess.run(['gh', 'api', '-X', 'PUT', f'repos/{owner}/{SHARE_REPO}/pages',
                    '-f', 'source[branch]=gh-pages', '-f', 'source[path]=/'], capture_output=True, timeout=60)

_BOOPIMG_RE = re.compile(r'/boopimg/([0-9a-f]{32}\.\w{1,5})')  # the exact name shape boopimg() serves; see paste_image() in cells.py
_DESC_RES = (re.compile(r'\*(?!\*)(.+?)\*$'), re.compile(r'>\s*(.+)$'))  # a lead-in written as *italics* or > a quote

def _pop_description(doc) -> str|None:
    "Take the line immediately under a notebook's '# ' title -- if it's written as *italics* or a > quote -- as the post's description, and REMOVE it from the notebook so it isn't also rendered in the body. Removal is the point: Quarto puts the description in the page's title block, and nbdev's own FrontmatterProc blanks that cell during a docs build, which is why the docs site never shows it twice; a share only runs color_cells, so without this the line appears both as the subtitle and again as the first paragraph. Only the one line counts, and only as the first non-blank thing after the heading -- anything further down is body text, not a summary. Blank lines between the two are ignored, and the line may sit in the title's own cell or the next markdown cell. Formatting is stripped, being markup for reading the notebook rather than part of the sentence. The doc passed in is the in-memory copy that gets written into the share directory, so the notebook on disk is untouched. Returns None (changing nothing) when there's no such line."
    cells = [c for c in doc.cells if c.cell_type == 'markdown']
    for i, c in enumerate(cells):
        src = c.source.lstrip()
        if not src.startswith('# '): continue
        rest = src.split('\n', 1)[1] if '\n' in src else ''
        target, lines = c, [l for l in rest.splitlines() if l.strip()]
        if not lines and i+1 < len(cells):
            target = cells[i+1]
            lines = [l for l in target.source.splitlines() if l.strip()]
        if not lines: return None
        for rx in _DESC_RES:
            if (m := rx.match(lines[0].strip())):
                # Drop just that line; the heading (and anything below) stays put. A cell left holding
                # nothing renders as nothing -- color_cells skips empty sources, so no stray ::: fence.
                keep = [l for l in target.source.splitlines() if l.strip() != lines[0].strip()]
                target.source = '\n'.join(keep).strip('\n')
                return m.group(1).strip()
        return None
    return None

def _docs_url() -> str:
    "The boopiter docs site's URL, read from the docs build's nbdev.yml -- the share index links back to it."
    import yaml
    proc = _repo_root()/'_proc'
    meta = yaml.safe_load((proc/'nbdev.yml').read_text()) if (proc/'nbdev.yml').exists() else {}
    return (meta.get('website') or {}).get('site-url') or ''

def _remote_shares(proj:Path) -> dict:
    "Every listed page already on gh-pages, as name -> card metadata, read from the `share.json` each publish leaves beside its page. This is what makes the site independent of which machine published it: the branch accumulates rendered pages from everywhere, and this reads back enough about the ones made elsewhere to keep them on the index. Reads through `git show` rather than a checkout, so nothing touches the working tree. An absent branch or unparseable sidecar yields nothing rather than raising -- the first share ever has no branch to read, and a page whose metadata can't be read is better dropped from the index than fatal to the publish."
    g = ['git', '-C', str(proj)]
    if subprocess.run(g + ['fetch', '-q', 'origin', 'gh-pages'], capture_output=True, timeout=180).returncode: return {}
    r = subprocess.run(g + ['ls-tree', '-r', '--name-only', 'FETCH_HEAD'], capture_output=True, text=True, timeout=60)
    out = {}
    for p in r.stdout.splitlines():
        if not p.endswith(f'/{_SIDECAR}') or p.count('/') != 1: continue
        j = subprocess.run(g + ['show', f'FETCH_HEAD:{p}'], capture_output=True, text=True, timeout=60)
        if j.returncode: continue
        try: out[p.split('/')[0]] = json.loads(j.stdout)
        except ValueError: continue
    return out

def _index_contents(remote:dict) -> str:
    "The listing's `contents:` block: a glob over the shares this machine can render, plus one inline entry per page that only exists on gh-pages. Quarto lets a listing item be given inline with its own metadata instead of being discovered from a file, which is the one mechanism that can put a card on the index for a page there is no local source for -- the alternative, a stub document, would render over the real page and replace it with the stub."
    import yaml
    items = [dict(path=f'{n}/', **{k: m[k] for k in ('title', 'description', 'date', 'image') if m.get(k)})
             for n, m in sorted(remote.items())]
    body = '\n    - "*/index.ipynb"'
    if items: body += '\n' + textwrap.indent(yaml.safe_dump(items, sort_keys=False, allow_unicode=True).rstrip(), '    ')
    return body

def _write_index(proj:Path, remote:dict) -> None:
    "Write the share site's index page, listing this machine's shares alongside every page already on gh-pages (see _index_contents)."
    (proj/'index.qmd').write_text(_INDEX_QMD.format(docs=_docs_url(), contents=_index_contents(remote)))

def _publish_additive(proj:Path, add:dict, drop:set) -> None:
    "Push the rendered _site onto gh-pages without deleting anything already there. `quarto publish gh-pages` cannot be used for this: it runs `git rm -r .` inside its worktree before copying the render in, so the branch ends up as exactly one machine's _site and every page published from anywhere else is destroyed. Here the worktree starts from the existing branch and the render is copied over the top, so a publish only ever adds or replaces its own directories. `add` carries the sidecars to write (the pages this run published, if listed); `drop` names the ones to delete, which is how a page flipped from public to unlisted stops appearing on other machines' indexes -- a copy alone can never remove a file. A worktree, not a checkout, so the project's own tree is untouched; a temp dir outside the project, so a stray render can't sweep it into the site."
    site = proj/'_site'
    for name, m in add.items():
        (site/name).mkdir(parents=True, exist_ok=True)
        (site/name/_SIDECAR).write_text(json.dumps(m, ensure_ascii=False))
    (site/'.nojekyll').touch()   # GitHub Pages serves the site as-is instead of running Jekyll over it
    g = ['git', '-C', str(proj)]
    subprocess.run(g + ['fetch', '-q', 'origin', 'gh-pages'], capture_output=True, timeout=180, check=True)
    tmp = Path(tempfile.mkdtemp(prefix='boop-publish-'))
    wt  = tmp/'gh-pages'         # a path that doesn't exist yet: `worktree add` refuses an existing dir
    subprocess.run(g + ['worktree', 'add', '-q', '--detach', str(wt), 'FETCH_HEAD'], check=True, timeout=120)
    try:
        shutil.copytree(site, wt, dirs_exist_ok=True)      # copies over, deletes nothing
        w = ['git', '-C', str(wt)]
        for name in drop: subprocess.run(w + ['rm', '-q', '--ignore-unmatch', f'{name}/{_SIDECAR}'], timeout=60)
        subprocess.run(w + ['add', '-Af', '.'], check=True, timeout=180)
        subprocess.run(w + ['commit', '-q', '--allow-empty', '-m', f"Publish {', '.join(sorted(add or drop))}"],
                       check=True, timeout=120)
        subprocess.run(w + ['push', '-q', 'origin', 'HEAD:gh-pages'], check=True, timeout=300)
    finally:
        subprocess.run(g + ['worktree', 'remove', '--force', str(wt)], capture_output=True, timeout=60)
        shutil.rmtree(tmp, ignore_errors=True)

def _write_share(nb_path, listed:bool, images_dir=None) -> tuple:
    "Put the processed notebook into the share project as <name>/index.ipynb, with a sibling _metadata.yml carrying its listed/unlisted state. Uses Quarto's per-directory metadata (the same mechanism a Quarto blog uses for posts/_metadata.yml) rather than injecting front matter, so the notebook itself is never rewritten and flipping public<->unlisted is a one-line file change. The title needs nothing: Quarto takes an .ipynb's title from its first '# ' heading, and color_cells deliberately leaves that H1 cell alone. Returns the share's directory name, and the card metadata another machine needs to list this page without ever seeing the notebook (see _SIDECAR)."
    import nbformat
    name = share_name(nb_path)
    d = SHARE_PROJECT/name
    d.mkdir(parents=True, exist_ok=True)
    doc = nbformat.read(str(nb_path), as_version=4)
    # Read title/description off the untouched cells: color_cells wraps markdown in ::: fences below,
    # which would hide a description living in the cell after the heading.
    titles = [c.source.lstrip()[2:].splitlines()[0].strip() for c in doc.cells
              if c.cell_type == 'markdown' and c.source.lstrip().startswith('# ')]
    desc = _pop_description(doc)   # also strips the line, so it isn't rendered twice
    img = None
    for c in doc.cells:
        # A pasted image is stored as /boopimg/<name>, a URL only a running boopiter can serve, so it
        # 404s on a shared page. Copy each referenced file in beside the notebook and point the
        # markdown at the bare name, which Quarto then treats as a resource and publishes with it.
        if c.cell_type == 'markdown' and images_dir:
            for n in dict.fromkeys(_BOOPIMG_RE.findall(c.source)):   # ordered, so the first is the thumbnail
                if (images_dir/n).exists():
                    shutil.copy(images_dir/n, d/n)
                    if img is None: img = n
            c.source = _BOOPIMG_RE.sub(lambda m: m.group(1) if (images_dir/m.group(1)).exists() else m.group(0),
                                       c.source)
        color_cells(c)
    # nbformat, NOT json.dump: nbformat stores a cell's source as a list of lines, and writing it
    # back as one string makes Quarto read the ::: fences as literal text, silently dropping the bars.
    nbformat.write(doc, str(d/'index.ipynb'))
    import yaml
    meta = {'date': date.today().isoformat()}
    # Quarto titles an .ipynb from its first '# ' heading (which is why color_cells leaves that cell
    # alone). With no such heading it would fall back to the filename 'index', identical for every
    # share, so name it after the notebook instead.
    if not titles: meta['title'] = Path(nb_path).stem
    if desc: meta['description'] = desc
    if not listed: meta['draft'] = True         # see 'drafts: unlinked' in _write_project_config
    # Dumped rather than hand-formatted: a description is arbitrary prose and can hold quotes, colons
    # or backticks that would otherwise need escaping by hand to stay valid YAML.
    (d/'_metadata.yml').write_text(yaml.safe_dump(meta, sort_keys=False, allow_unicode=True))
    # The card other machines will draw for this page (see _SIDECAR). The same facts as _metadata.yml, but
    # resolved rather than left to Quarto: the title it would infer from the H1, and an image path relative
    # to the site root rather than to the document, since an inline listing entry is read from the index.
    card = {'title': titles[0] if titles else Path(nb_path).stem, 'date': meta['date']}
    if desc: card['description'] = desc
    if img: card['image'] = f'{name}/{img}'
    return name, card

def share_notebook(nb_path, listed:bool=True, images_dir=None) -> str:
    "Render `nb_path` into the share site and publish it, returning the public URL. `listed` puts it on the site's index; unlisted pages are still rendered and reachable by URL, just not indexed -- and re-sharing with the other choice flips that, since the state lives in a file that gets rewritten. Overwrites in place: the directory is named from the notebook's identity (see share_slug), so a correction updates the page a recipient already has the link to. Renders locally and pushes only the built site to the gh-pages branch, which is why notebook sources never leave this machine. The push is ours rather than `quarto publish gh-pages` because that command clears the branch before copying its render in, which deletes every page published from a different machine; see _publish_additive. Pages made elsewhere are read back off the branch and listed beside this machine's, so the site reads the same no matter which machine published last, and publishing only ever adds -- see _remote_shares. Returns as soon as the push lands; GitHub Pages then takes roughly a minute to rebuild."
    proj   = _ensure_share_project()
    remote = _remote_shares(proj)
    name, card = _write_share(nb_path, listed, images_dir)
    remote.pop(name, None)   # rendered from source here, so an inline card too would list it twice
    _write_index(proj, remote)
    subprocess.run(['quarto', 'render'], cwd=proj, capture_output=True, timeout=900, check=True)
    # An unlisted share publishes its page but no sidecar, and drops any sidecar a previous public share of
    # the same notebook left behind -- otherwise flipping public -> unlisted would keep the card on every
    # other machine's index, which is the one thing unlisted is supposed to prevent.
    _publish_additive(proj, {name: card} if listed else {}, set() if listed else {name})
    return f'https://{_gh_owner()}.github.io/{SHARE_REPO}/{name}/'